<a href="https://colab.research.google.com/github/hafizihsani/data-science-2026/blob/main/Pertemuan2_Muhamad_Hafiz_ihsani_250401020155.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np

# Menggunakan link raw agar bisa dibaca langsung oleh pandas
url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# Menampilkan 5 data teratas untuk memastikan data termuat
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [22]:
# Menampilkan jumlah baris dan kolom
rows, cols = df.shape
print(f"Jumlah Baris: {rows}")
print(f"Jumlah Kolom: {cols}")

# Menampilkan daftar nama kolom
print("Daftar Kolom:", df.columns.tolist())

Jumlah Baris: 891
Jumlah Kolom: 12
Daftar Kolom: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']


Temuan: Dataset ini memiliki 891 baris dan 12 kolom. Kolom yang tersedia mencakup informasi demografis (Age, Sex), status perjalanan (Pclass, Fare), dan target utama (Survived).

---



In [24]:
# Hitung jumlah missing values per kolom
missing_count = df.isnull().sum()

# Hitung proporsi dalam persen
missing_percent = (missing_count / len(df)) * 100
missing_data = pd.DataFrame({
    'Jumlah Missing': missing_count,
    'Persentase (%)': missing_percent
})

print(missing_data[missing_data['Jumlah Missing'] > 0])


          Jumlah Missing  Persentase (%)
Age                  177       19.865320
Cabin                687       77.104377
Embarked               2        0.224467


Temuan: Kolom Age, Cabin, dan Embarked memiliki data yang hilang. Kolom Cabin memiliki missing value terbanyak (lebih dari 70%), yang berarti kolom ini mungkin kurang efektif jika digunakan dalam model tanpa pra-pemrosesan yang kuat.

---



In [29]:
persentase_numpy = np.mean(df['Survived'].values) * 100

print(f"Persentase Selamat (NumPy): {persentase_numpy:.2f}%")

Persentase Selamat (NumPy): 38.38%


Temuan: Sekitar 38.38% penumpang Titanic selamat dari tragedi tersebut.

---



In [30]:
# 1. Melakukan filter: Jenis kelamin wanita DAN kelas penumpang 1
wanita_kelas1 = df[(df['Sex'] == 'female') & (df['Pclass'] == 1)]

# 2. Menghitung jumlah baris hasil filter
jumlah_wanita_k1 = len(wanita_kelas1)

# 3. Menghitung rata-rata usia pada hasil filter tersebut
# .mean() secara otomatis mengabaikan nilai NaN (missing values) pada kolom Age
rata_usia_wanita_k1 = wanita_kelas1['Age'].mean()

print(f"Jumlah penumpang wanita dari kelas 1: {jumlah_wanita_k1} orang")
print(f"Rata-rata usia penumpang wanita kelas 1: {rata_usia_wanita_k1:.2f} tahun")

Jumlah penumpang wanita dari kelas 1: 94 orang
Rata-rata usia penumpang wanita kelas 1: 34.61 tahun


"Berdasarkan hasil pemfilteran data, terdapat 94 penumpang wanita yang terdaftar di kelas 1. Rata-rata usia mereka adalah sekitar 34.61 tahun. Analisis ini menunjukkan profil demografis penumpang kelas atas pada saat itu, di mana usia rata-ratanya cenderung sudah matang."


---



In [31]:
# 1. Mengelompokkan data berdasarkan 'Pclass' dan mengambil rata-rata dari kolom 'Survived'
tingkat_selamat_per_kelas = df.groupby('Pclass')['Survived'].mean()

# 2. Mengonversi ke persentase agar lebih mudah dibaca (opsional)
persentase_per_kelas = tingkat_selamat_per_kelas * 100

print("Tingkat Keselamatan per Kelas Penumpang:")
print(persentase_per_kelas)

# 3. Mencari kelas dengan tingkat keselamatan tertinggi secara otomatis
kelas_tertinggi = tingkat_selamat_per_kelas.idxmax()
nilai_tertinggi = tingkat_selamat_per_kelas.max() * 100

print(f"\nKelas dengan tingkat keselamatan paling tinggi adalah Kelas {kelas_tertinggi} dengan {nilai_tertinggi:.2f}%")

Tingkat Keselamatan per Kelas Penumpang:
Pclass
1    62.962963
2    47.282609
3    24.236253
Name: Survived, dtype: float64

Kelas dengan tingkat keselamatan paling tinggi adalah Kelas 1 dengan 62.96%


"Hasil analisis menggunakan groupby menunjukkan korelasi yang kuat antara kelas sosial (ekonomi) dengan peluang keselamatan. Penumpang di Kelas 1 memiliki tingkat keselamatan tertinggi (sekitar 62.96%), jauh melampaui Kelas 2 dan Kelas 3. Hal ini menunjukkan adanya prioritas evakuasi atau akses yang lebih baik ke sekoci bagi penumpang kelas atas."


---



In [32]:
# 1. Membuat kategori baru 'IsChild' (Anak-anak jika usia < 16 tahun)
df['AgeGroup'] = np.where(df['Age'] < 16, 'Child', 'Adult')

# 2. Menghitung rata-rata keselamatan berdasarkan kategori tersebut
survival_age_group = df.groupby('AgeGroup')['Survived'].mean() * 100

print("Tingkat Keselamatan: Anak-anak (<16 thn) vs Dewasa:")
print(survival_age_group)

# 3. Analisis tambahan: Melihat jumlah penumpang per kategori
print("\nJumlah Penumpang per Kategori:")
print(df['AgeGroup'].value_counts())

Tingkat Keselamatan: Anak-anak (<16 thn) vs Dewasa:
AgeGroup
Adult    36.262376
Child    59.036145
Name: Survived, dtype: float64

Jumlah Penumpang per Kategori:
AgeGroup
Adult    808
Child     83
Name: count, dtype: int64


"Saya menganalisis tingkat keselamatan berdasarkan kelompok usia dengan membagi penumpang menjadi dua kategori: Child (di bawah 16 tahun) dan Adult. Hasilnya menunjukkan bahwa anak-anak memiliki tingkat keselamatan yang signifikan lebih tinggi (sekitar 59%) dibandingkan orang dewasa (sekitar 36%).

Temuan ini memperkuat bukti adanya kebijakan non-formal 'women and children first' selama proses evakuasi kapal Titanic. Meskipun jumlah anak-anak jauh lebih sedikit, prioritas yang diberikan kepada mereka sangat berdampak pada peluang hidup mereka."